# Phase 3: Human-in-the-Loop Active Learning
# Medical Chatbot - Uncertainty-Based Expert Review System

This notebook implements active learning techniques to identify low-confidence model responses that need expert review.

**Objectives:**
1. Calculate uncertainty scores for model predictions
2. Identify samples requiring expert feedback
3. Collect and process expert annotations
4. Create feedback pipeline for continuous improvement

**Uncertainty Methods:**
- Entropy-based scoring
- Logit margin analysis
- Response confidence thresholds

## 1. Setup and Imports

In [1]:
# Install required packages
!pip install -q transformers datasets torch numpy scipy pandas accelerate peft bitsandbytes ipywidgets

In [2]:
import torch
import numpy as np
import pandas as pd
import json
import os
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Tuple
from scipy.stats import entropy
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, PeftConfig
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import warnings
warnings.filterwarnings('ignore')

print(" All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"ipywidgets version: {widgets.__version__}")

 All imports successful!
PyTorch version: 2.8.0+cu128
CUDA available: True
ipywidgets version: 8.1.7


In [3]:
# Configuration - ADJUSTED FOR FINE-TUNED MODEL + INTERACTIVE REVIEW
CONFIG = {
    'base_model': 'mistralai/Mistral-7B-Instruct-v0.3',
    'adapter_path': './model_output',  # Path to Phase 2 trained model
    'test_data_path': './processed_data/processed_test.json',
    'output_dir': './expert_feedback',
    'uncertainty_threshold': 0.25,  # BALANCED: 40-60% review rate for interactive review
    'medical_risk_threshold': 0.01,  # Even lower for high-risk medical keywords
    'top_k_sampling': 50,  # For uncertainty calculation
    'batch_size': 10,  # Increased for interactive review
    'max_length': 512,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    # Medical safety keywords that require extra scrutiny
    'high_risk_keywords': [
        'ibuprofen', 'aspirin', 'acetaminophen', 'paracetamol',
        'medication', 'medicine', 'drug', 'dosage', 'dose', 'mg', 'ml',
        'chest pain', 'heart attack', 'stroke', 'bleeding', 'overdose',
        'prescription', 'antibiotic', 'pain killer', 'blood pressure',
        'insulin', 'emergency', 'should i take', 'how much', 'how many'
    ]
}

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)
print(f" Output directory: {CONFIG['output_dir']}")
print(f" Device: {CONFIG['device']}")
print(f" Uncertainty Threshold: {CONFIG['uncertainty_threshold']} (balanced for interactive review)")
print(f"Medical Risk Threshold: {CONFIG['medical_risk_threshold']} (high-risk keywords)")

 Output directory: ./expert_feedback
 Device: cuda
 Uncertainty Threshold: 0.25 (balanced for interactive review)
Medical Risk Threshold: 0.01 (high-risk keywords)


## 2. Load Trained Model from Phase 2

In [4]:
def load_trained_model(base_model_name: str, adapter_path: str, device: str):
    """
    Load the fine-tuned Mistral-7B model with LoRA adapters from Phase 2.
    
    Args:
        base_model_name: HuggingFace model identifier
        adapter_path: Path to saved LoRA adapters
        device: 'cuda' or 'cpu'
    
    Returns:
        model, tokenizer
    """
    print(" Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(adapter_path)
    
    print(" Loading base model with quantization...")
    # Use 4-bit quantization for memory efficiency
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    
    print("Loading LoRA adapters...")
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()
    
    print("Model loaded successfully!")
    return model, tokenizer

In [5]:
# Load the trained model
model, tokenizer = load_trained_model(
    CONFIG['base_model'],
    CONFIG['adapter_path'],
    CONFIG['device']
)

print(f"\n Model Info:")
print(f"   Base Model: {CONFIG['base_model']}")
print(f"   Adapter Path: {CONFIG['adapter_path']}")
print(f"   Vocab Size: {len(tokenizer)}")

 Loading tokenizer...
 Loading base model with quantization...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading LoRA adapters...
Model loaded successfully!

 Model Info:
   Base Model: mistralai/Mistral-7B-Instruct-v0.3
   Adapter Path: ./model_output
   Vocab Size: 32768


## 3. Uncertainty Sampling Implementation

We implement multiple uncertainty metrics:
1. **Entropy**: Measures randomness in probability distribution
2. **Logit Margin**: Difference between top-2 predictions
3. **Max Probability**: Confidence of the most likely token

In [6]:
def calculate_uncertainty_score(model, tokenizer, question: str, max_length: int = 512) -> Dict:
    """
    Calculate uncertainty metrics for a model's response with MEDICAL RISK DETECTION.
    
    ENHANCED FOR FINE-TUNED MODELS:
    - Detects high-risk medical keywords (dosage, medication, etc.)
    - Applies uncertainty boost for medical safety
    - Uses multiple uncertainty methods
    - Adjusted thresholds for overconfident models
    
    Methods:
    - Entropy: Higher entropy = more uncertain
    - Logit Margin: Smaller margin = more uncertain  
    - Max Probability: Lower max prob = more uncertain
    - Medical Risk: Keyword detection with uncertainty boost
    
    Args:
        model: The fine-tuned model
        tokenizer: Model tokenizer
        question: Input medical question
        max_length: Maximum generation length
    
    Returns:
        Dictionary with uncertainty metrics and generated response
    """
    # Format input with chat template
    messages = [
        {"role": "system", "content": "You are a helpful medical assistant. Provide accurate and clear medical information."},
        {"role": "user", "content": question}
    ]
    
    # Tokenize input
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    # Generate response with output scores
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            return_dict_in_generate=True,
            output_scores=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode generated text
    generated_ids = outputs.sequences[0][inputs.input_ids.shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    # Calculate uncertainty metrics from logits
    logits_list = outputs.scores  # List of tensors, one per generated token
    
    entropies = []
    max_probs = []
    logit_margins = []
    
    for logits in logits_list:
        probs = torch.softmax(logits[0], dim=-1)
        
        # Entropy calculation
        token_entropy = entropy(probs.cpu().numpy())
        entropies.append(token_entropy)
        
        # Max probability
        max_prob = torch.max(probs).item()
        max_probs.append(max_prob)
        
        # Logit margin (difference between top 2)
        top2_logits = torch.topk(logits[0], k=2).values
        margin = (top2_logits[0] - top2_logits[1]).item()
        logit_margins.append(margin)
    
    # Aggregate metrics
    avg_entropy = np.mean(entropies)
    avg_max_prob = np.mean(max_probs)
    avg_margin = np.mean(logit_margins)
    
    # Base uncertainty score (0-1, higher = more uncertain)
    # Re-weighted for fine-tuned models that are overconfident
    base_uncertainty = (
        0.5 * (1 - avg_max_prob) +   # Low confidence (increased weight)
        0.3 * (avg_entropy / 8) +     # High entropy (adjusted normalization)
        0.2 * (1 / (1 + avg_margin))  # Low margin
    )
    
    # MEDICAL RISK DETECTION: Check for high-risk keywords
    question_lower = question.lower()
    response_lower = generated_text.lower()
    
    # Detect medical risk keywords
    medical_risk_detected = False
    risk_keywords_found = []
    
    for keyword in CONFIG['high_risk_keywords']:
        if keyword in question_lower or keyword in response_lower:
            medical_risk_detected = True
            risk_keywords_found.append(keyword)
    
    # Apply uncertainty boost for medical risk
    uncertainty_score = base_uncertainty
    uncertainty_boost = 0.0
    
    if medical_risk_detected:
        # Boost uncertainty for medical safety
        # More keywords = higher boost
        keyword_multiplier = min(len(risk_keywords_found) * 0.15, 0.5)
        
        # Specific high-risk patterns get extra boost
        if any(kw in question_lower for kw in ['dosage', 'how much', 'how many', 'mg', 'ml']):
            uncertainty_boost = 0.4  # Strong boost for dosage questions
        elif any(kw in question_lower for kw in ['should i take', 'medication', 'drug']):
            uncertainty_boost = 0.35  # Medium-high boost for medication questions
        elif any(kw in question_lower for kw in ['chest pain', 'emergency', 'overdose']):
            uncertainty_boost = 0.45  # Very high boost for emergency keywords
        else:
            uncertainty_boost = keyword_multiplier  # Standard boost based on keyword count
        
        # Apply boost and cap at 1.0
        uncertainty_score = min(base_uncertainty + uncertainty_boost, 1.0)
    
    return {
        'question': question,
        'response': generated_text,
        'uncertainty_score': float(uncertainty_score),
        'base_uncertainty': float(base_uncertainty),
        'uncertainty_boost': float(uncertainty_boost),
        'medical_risk_detected': medical_risk_detected,
        'risk_keywords': risk_keywords_found,
        'avg_entropy': float(avg_entropy),
        'avg_confidence': float(avg_max_prob),
        'avg_margin': float(avg_margin),
        'num_tokens': len(logits_list),
        'timestamp': datetime.now().isoformat()
    }

### Test Medical Risk Detection

Quick test to verify the enhanced uncertainty calculation works correctly.

In [ ]:
# Test medical risk detection with sample questions
print("TESTING MEDICAL RISK DETECTION\n")
print("Testing enhanced uncertainty with medical keyword detection...")
print("="*80)

test_questions = [
    "What are the symptoms of diabetes?",  # Low risk
    "How much ibuprofen should I take for headache?",  # HIGH RISK - dosage
    "What causes high blood pressure?"  # Medium risk
]

for i, q in enumerate(test_questions, 1):
    print(f"\n[Test {i}] {q}")
    
    # Simulate detection (without running model)
    q_lower = q.lower()
    risk_detected = any(kw in q_lower for kw in CONFIG['high_risk_keywords'])
    
    if risk_detected:
        matched_keywords = [kw for kw in CONFIG['high_risk_keywords'] if kw in q_lower]
        print(f"   Medical Risk: DETECTED")
        print(f"   Keywords: {matched_keywords}")
        
        # Check for high-risk patterns
        if any(kw in q_lower for kw in ['dosage', 'how much', 'how many', 'mg', 'ml']):
            print(f"   Boost: 0.4 (DOSAGE QUESTION)")
        elif any(kw in q_lower for kw in ['should i take', 'medication']):
            print(f"   Boost: 0.35 (MEDICATION QUESTION)")
        else:
            print(f"   Boost: 0.15-0.5 (based on keyword count)")
    else:
        print(f"   Medical Risk: None")
        print(f"   Boost: 0.0 (standard uncertainty)")

print("\n" + "="*80)
print("Medical risk detection is working! \n")

🧪 TESTING MEDICAL RISK DETECTION

Testing enhanced uncertainty with medical keyword detection...

[Test 1] What are the symptoms of diabetes?
  ✅ Medical Risk: None
  ⚡ Boost: 0.0 (standard uncertainty)

[Test 2] How much ibuprofen should I take for headache?
  🚨 Medical Risk: DETECTED
  📌 Keywords: ['ibuprofen', 'should i take', 'how much']
  ⚡ Boost: 0.4 (DOSAGE QUESTION)

[Test 3] What causes high blood pressure?
  🚨 Medical Risk: DETECTED
  📌 Keywords: ['blood pressure']
  ⚡ Boost: 0.15-0.5 (based on keyword count)

✅ Medical risk detection is working! Ready to run full pipeline.



In [8]:
def identify_low_confidence_samples(
    results: List[Dict],
    threshold: float = 0.7
) -> Tuple[List[Dict], List[Dict]]:
    """
    Separate samples into high and low confidence groups.
    
    Args:
        results: List of uncertainty calculation results
        threshold: Uncertainty threshold (higher = needs review)
    
    Returns:
        (low_confidence_samples, high_confidence_samples)
    """
    low_confidence = []
    high_confidence = []
    
    for result in results:
        if result['uncertainty_score'] > threshold:
            low_confidence.append(result)
        else:
            high_confidence.append(result)
    
    # Sort by uncertainty score (highest first)
    low_confidence.sort(key=lambda x: x['uncertainty_score'], reverse=True)
    
    return low_confidence, high_confidence

## 4. Expert Feedback Collection System

Simple annotation interface for medical experts to rate model responses.

In [ ]:
def collect_expert_feedback(
    sample: Dict,
    rating: int,
    corrected_response: str = None,
    expert_notes: str = None
) -> Dict:
    """
    Collect and structure expert feedback for a sample.
    
    Rating Scale:
    5 - Excellent: Accurate, complete, well-explained
    4 - Good: Mostly accurate, minor improvements needed
    3 - Acceptable: Some inaccuracies, needs revision
    2 - Poor: Significant issues, major revision needed
    1 - Unacceptable: Incorrect or dangerous information
    
    Args:
        sample: Sample dictionary with question and response
        rating: Expert rating (1-5)
        corrected_response: Expert's corrected version (optional)
        expert_notes: Additional notes from expert (optional)
    
    Returns:
        Annotated sample dictionary
    """
    # Convert numpy types to native Python types for JSON serialization
    annotation = {
        'sample_id': sample.get('sample_id', hash(sample['question'])),
        'question': sample['question'],
        'model_response': sample['response'],
        'uncertainty_score': float(sample['uncertainty_score']),
        'expert_rating': int(rating),   
        'corrected_response': corrected_response,
        'expert_notes': expert_notes,
        'annotation_timestamp': datetime.now().isoformat(),
        'needs_retraining': bool(rating <= 3)  
    }
    
    return annotation

In [ ]:
def create_review_batch(
    low_confidence_samples: List[Dict],
    batch_size: int = 10,
    output_path: str = None
) -> List[Dict]:
    """
    Prepare a batch of samples for expert review.
    
    Args:
        low_confidence_samples: List of uncertain predictions
        batch_size: Number of samples per batch
        output_path: Path to save review batch
    
    Returns:
        List of samples formatted for review
    """
    # Select top uncertain samples
    review_batch = low_confidence_samples[:batch_size]
    
    # Add sample IDs and review metadata
    for idx, sample in enumerate(review_batch):
        sample['sample_id'] = f"review_{datetime.now().strftime('%Y%m%d')}_{idx:03d}"
        sample['review_status'] = 'pending'
    
    # Save if output path provided
    if output_path:
        with open(output_path, 'w') as f:
            for sample in review_batch:
                f.write(json.dumps(sample) + '\n')
        print(f"Review batch saved: {output_path}")
    
    return review_batch

In [ ]:
def save_feedback_data(
    annotations: List[Dict],
    output_path: str
):
    """
    Save expert feedback annotations in JSONL format.
    
    Args:
        annotations: List of annotated samples
        output_path: Output file path
    """
    with open(output_path, 'w') as f:
        for annotation in annotations:
            f.write(json.dumps(annotation) + '\n')
    
    print(f" Feedback data saved: {output_path}")
    print(f"   Total annotations: {len(annotations)}")
    
    # Calculate statistics
    ratings = [a['expert_rating'] for a in annotations]
    needs_retraining = sum(1 for a in annotations if a['needs_retraining'])
    
    print(f"   Average rating: {np.mean(ratings):.2f}")
    print(f"   Samples needing retraining: {needs_retraining}/{len(annotations)}")

## 5. Active Learning Pipeline

Complete workflow for uncertainty-based sample selection and expert review.

In [ ]:
def active_learning_pipeline(
    model,
    tokenizer,
    questions: List[str],
    uncertainty_threshold: float = 0.7,
    batch_size: int = 10,
    output_dir: str = './expert_feedback'
) -> Dict:
    """
    Execute complete active learning pipeline.
    
    Steps:
    1. Generate predictions with uncertainty scores
    2. Identify low-confidence samples
    3. Create review batches for experts
    4. Save results for annotation
    
    Args:
        model: Trained model
        tokenizer: Model tokenizer
        questions: List of medical questions
        uncertainty_threshold: Threshold for expert review
        batch_size: Review batch size
        output_dir: Directory for outputs
    
    Returns:
        Pipeline statistics and results
    """
    print(" Starting Active Learning Pipeline...\n")
    
    # Step 1: Calculate uncertainty scores
    print(f"Step 1: Calculating uncertainty for {len(questions)} questions...")
    uncertainty_results = []
    
    for idx, question in enumerate(questions):
        print(f"   Processing {idx+1}/{len(questions)}", end='\r')
        result = calculate_uncertainty_score(model, tokenizer, question)
        uncertainty_results.append(result)
    
    print(f"\n Uncertainty calculation complete!")
    
    # Step 2: Identify low-confidence samples
    print(f"\n  Step 2: Identifying low-confidence samples...")
    low_conf, high_conf = identify_low_confidence_samples(
        uncertainty_results,
        threshold=uncertainty_threshold
    )
    
    print(f"   Low confidence (need review): {len(low_conf)}")
    print(f"   High confidence (no review): {len(high_conf)}")
    print(f"   Review rate: {len(low_conf)/len(questions)*100:.1f}%")
    
    # Step 3: Create review batch
    print(f"\n  Step 3: Creating expert review batch...")
    pending_path = os.path.join(output_dir, 'pending_samples.jsonl')
    review_batch = create_review_batch(
        low_conf,
        batch_size=min(batch_size, len(low_conf)),
        output_path=pending_path
    )
    
    # Step 4: Save all results
    print(f"\n Step 4: Saving pipeline results...")
    
    # Save all uncertainty results
    all_results_path = os.path.join(output_dir, 'all_uncertainty_results.jsonl')
    with open(all_results_path, 'w') as f:
        for result in uncertainty_results:
            f.write(json.dumps(result) + '\n')
    print(f"    All results: {all_results_path}")
    
    # Save high confidence samples (no review needed)
    high_conf_path = os.path.join(output_dir, 'high_confidence_samples.jsonl')
    with open(high_conf_path, 'w') as f:
        for result in high_conf:
            f.write(json.dumps(result) + '\n')
    print(f"    High confidence: {high_conf_path}")
    
    # Compile statistics
    stats = {
        'total_samples': len(questions),
        'low_confidence_count': len(low_conf),
        'high_confidence_count': len(high_conf),
        'review_rate': len(low_conf) / len(questions),
        'avg_uncertainty': float(np.mean([r['uncertainty_score'] for r in uncertainty_results])),
        'avg_confidence': float(np.mean([r['avg_confidence'] for r in uncertainty_results])),
        'review_batch_size': len(review_batch),
        'timestamp': datetime.now().isoformat()
    }
    
    # Save statistics
    stats_path = os.path.join(output_dir, 'pipeline_statistics.json')
    with open(stats_path, 'w') as f:
        json.dump(stats, f, indent=2)
    print(f"   Statistics: {stats_path}")
    
    print("\n" + "="*60)
    print(" ACTIVE LEARNING PIPELINE COMPLETE!")
    print("="*60)
    
    return stats

## 6. Demo and Testing

Run the active learning pipeline with sample medical questions.

In [ ]:
# Sample medical questions for testing
sample_questions = [
    "What are the common symptoms of type 2 diabetes?",
    "How does hypertension affect cardiovascular health?",
    "What is the difference between viral and bacterial infections?",
    "Explain the mechanism of action of beta blockers.",
    "What are the risk factors for developing osteoporosis?",
    "How is asthma diagnosed and managed?",
    "What causes rheumatoid arthritis and how is it treated?",
    "Describe the stages of chronic kidney disease.",
    "What are the warning signs of a stroke?",
    "How does chemotherapy work to treat cancer?",
    "What is the role of insulin in glucose metabolism?",
    "Explain the difference between Alzheimer's disease and dementia.",
    "What are the contraindications for using NSAIDs?",
    "How does vaccination provide immunity?",
    "What lifestyle changes can help manage high cholesterol?",
    "How much ibuprofen should I take for headache?"
]

print(f" Testing with {len(sample_questions)} medical questions")
print("\nSample questions:")
for i, q in enumerate(sample_questions[:5], 1):
    print(f"{i}. {q}")

📋 Testing with 16 medical questions

Sample questions:
1. What are the common symptoms of type 2 diabetes?
2. How does hypertension affect cardiovascular health?
3. What is the difference between viral and bacterial infections?
4. Explain the mechanism of action of beta blockers.
5. What are the risk factors for developing osteoporosis?


In [14]:
# Run the active learning pipeline
pipeline_stats = active_learning_pipeline(
    model=model,
    tokenizer=tokenizer,
    questions=sample_questions,
    uncertainty_threshold=CONFIG['uncertainty_threshold'],
    batch_size=CONFIG['batch_size'],
    output_dir=CONFIG['output_dir']
)

🚀 Starting Active Learning Pipeline...

📊 Step 1: Calculating uncertainty for 16 questions...
   Processing 16/16
✅ Uncertainty calculation complete!

📊 Step 2: Identifying low-confidence samples...
   Low confidence (need review): 6
   High confidence (no review): 10
   Review rate: 37.5%

📊 Step 3: Creating expert review batch...
✅ Review batch saved: ./expert_feedback/pending_samples.jsonl

📊 Step 4: Saving pipeline results...
   ✅ All results: ./expert_feedback/all_uncertainty_results.jsonl
   ✅ High confidence: ./expert_feedback/high_confidence_samples.jsonl
   ✅ Statistics: ./expert_feedback/pipeline_statistics.json

✅ ACTIVE LEARNING PIPELINE COMPLETE!


In [15]:
# Display pipeline statistics
print("\n📊 PIPELINE STATISTICS")
print("="*60)
print(f"Total samples processed: {pipeline_stats['total_samples']}")
print(f"Low confidence (need review): {pipeline_stats['low_confidence_count']}")
print(f"High confidence (no review): {pipeline_stats['high_confidence_count']}")
print(f"Review rate: {pipeline_stats['review_rate']*100:.1f}%")
print(f"Average uncertainty score: {pipeline_stats['avg_uncertainty']:.3f}")
print(f"Average confidence: {pipeline_stats['avg_confidence']:.3f}")
print(f"Review batch size: {pipeline_stats['review_batch_size']}")
print("="*60)


📊 PIPELINE STATISTICS
Total samples processed: 16
Low confidence (need review): 6
High confidence (no review): 10
Review rate: 37.5%
Average uncertainty score: 0.228
Average confidence: 0.947
Review batch size: 6


### Example: View Low Confidence Samples

In [ ]:
# Load and display pending samples WITH MEDICAL RISK INFO
pending_path = os.path.join(CONFIG['output_dir'], 'pending_samples.jsonl')

if os.path.exists(pending_path):
    pending_samples = []
    with open(pending_path, 'r') as f:
        for line in f:
            pending_samples.append(json.loads(line))
    
    print(f"\n TOP {len(pending_samples)} SAMPLES NEEDING EXPERT REVIEW")
    print("="*80)
    
    for idx, sample in enumerate(pending_samples[:3], 1):  # Show top 3
        print(f"\n[Sample {idx}] ID: {sample['sample_id']}")
        print(f"Uncertainty Score: {sample['uncertainty_score']:.3f}")
        
        # Show medical risk information if available
        if 'medical_risk_detected' in sample:
            print(f" Medical Risk: {'DETECTED' if sample['medical_risk_detected'] else 'None'}")

            if sample['medical_risk_detected']:
                print(f"   Risk Keywords: {sample.get('risk_keywords', [])}")
                print(f"   Base Uncertainty: {sample.get('base_uncertainty', 0):.3f}")
                print(f"   Uncertainty Boost: +{sample.get('uncertainty_boost', 0):.3f}")
        
        print(f"Confidence: {sample['avg_confidence']:.3f}")
        print(f"\nQuestion: {sample['question']}")
        print(f"\nModel Response: {sample['response'][:200]}...")
        print("\n" + "-"*80)
else:
    print(" No pending samples file found")


🔍 TOP 6 SAMPLES NEEDING EXPERT REVIEW

[Sample 1] ID: review_20251009_000
Uncertainty Score: 0.525
🚨 Medical Risk: DETECTED
   Risk Keywords: ['ibuprofen', 'acetaminophen', 'medication', 'drug', 'bleeding', 'blood pressure']
   Base Uncertainty: 0.025
   Uncertainty Boost: +0.500
Confidence: 0.957

Question: What are the contraindications for using NSAIDs?

Model Response: NSAIDs (non-steroidal anti-inflammatory drugs) have several contraindications. Here are some of them:

1. Allergies: If you are allergic to any NSAID, do not take it.
2. Kidney problems: NSAIDs can wo...

--------------------------------------------------------------------------------

[Sample 2] ID: review_20251009_001
Uncertainty Score: 0.427
🚨 Medical Risk: DETECTED
   Risk Keywords: ['ibuprofen', 'medication', 'mg', 'blood pressure', 'should i take', 'how much']
   Base Uncertainty: 0.027
   Uncertainty Boost: +0.400
Confidence: 0.955

Question: How much ibuprofen should I take for headache?

Model Response: for

### Demonstrate Expert Feedback Collection

In [ ]:
# Simulate expert feedback for demonstration
print(" SIMULATING EXPERT FEEDBACK COLLECTION\n")
print("In production, experts would review samples through an annotation interface.")
print("For this demo, we'll create mock annotations.\n")

# Load pending samples
pending_path = os.path.join(CONFIG['output_dir'], 'pending_samples.jsonl')
mock_annotations = []

if os.path.exists(pending_path):
    with open(pending_path, 'r') as f:
        pending_samples = [json.loads(line) for line in f]
    
    # Create mock expert feedback
    for idx, sample in enumerate(pending_samples):
        # Simulate varying expert ratings
        mock_rating = np.random.choice([2, 3, 4], p=[0.3, 0.5, 0.2])  # Weighted towards middle ratings
        
        annotation = collect_expert_feedback(
            sample=sample,
            rating=mock_rating,
            corrected_response=f"[Expert corrected version for sample {idx+1}]" if mock_rating <= 3 else None,
            expert_notes=f"Sample shows {'significant' if mock_rating <= 2 else 'minor'} issues requiring attention."
        )
        mock_annotations.append(annotation)
        
        print(f"Sample {idx+1}: Rating = {mock_rating}/5, Needs Retraining = {annotation['needs_retraining']}")
    
    # Save annotations
    annotations_path = os.path.join(CONFIG['output_dir'], 'annotations.jsonl')
    save_feedback_data(mock_annotations, annotations_path)
    
else:
    print(" No pending samples available for annotation")

📝 SIMULATING EXPERT FEEDBACK COLLECTION

In production, experts would review samples through an annotation interface.
For this demo, we'll create mock annotations.

Sample 1: Rating = 2/5, Needs Retraining = True
Sample 2: Rating = 3/5, Needs Retraining = True
Sample 3: Rating = 3/5, Needs Retraining = True
Sample 4: Rating = 3/5, Needs Retraining = True
Sample 5: Rating = 4/5, Needs Retraining = False
Sample 6: Rating = 2/5, Needs Retraining = True
✅ Feedback data saved: ./expert_feedback/annotations.jsonl
   Total annotations: 6
   Average rating: 2.83
   Samples needing retraining: 5/6


### 🎯 Interactive Expert Review Interface

Use this in-notebook interface to review uncertain samples directly in Jupyter.

In [18]:
class InteractiveExpertReview:
    """
    Interactive widget-based expert review interface for Jupyter notebooks.
    Allows experts to review uncertain medical samples directly in the notebook.
    """
    
    def __init__(self, samples: List[Dict], output_dir: str = './expert_feedback'):
        """
        Initialize the interactive review interface.
        
        Args:
            samples: List of uncertain samples to review
            output_dir: Directory to save expert annotations
        """
        self.samples = samples
        self.output_dir = output_dir
        self.current_index = 0
        self.annotations = []
        
        # Create widgets
        self._create_widgets()
        
    def _create_widgets(self):
        """Create all UI widgets for the review interface."""
        
        # Header with sample info
        self.header = widgets.HTML(
            value="<h3>🏥 Medical Sample Review Interface</h3>"
        )
        
        # Progress bar
        self.progress = widgets.IntProgress(
            value=0,
            min=0,
            max=len(self.samples),
            description='Progress:',
            bar_style='info',
            style={'bar_color': '#4CAF50'},
            layout=widgets.Layout(width='100%')
        )
        
        self.progress_text = widgets.HTML(
            value=f"<b>Sample 0 of {len(self.samples)}</b>"
        )
        
        # Sample information display
        self.sample_info = widgets.HTML(
            value="<i>Click 'Start Review' to begin</i>",
            layout=widgets.Layout(width='100%', border='1px solid #ddd', padding='10px')
        )
        
        # Question display
        self.question_box = widgets.Textarea(
            value='',
            description='Question:',
            disabled=True,
            layout=widgets.Layout(width='100%', height='80px'),
            style={'description_width': '100px'}
        )
        
        # Model response display
        self.response_box = widgets.Textarea(
            value='',
            description='Model Response:',
            disabled=True,
            layout=widgets.Layout(width='100%', height='120px'),
            style={'description_width': '100px'}
        )
        
        # Uncertainty info
        self.uncertainty_info = widgets.HTML(
            value='',
            layout=widgets.Layout(width='100%', padding='5px')
        )
        
        # Rating buttons
        self.rating_buttons = widgets.RadioButtons(
            options=[
                ('✅ Safe - Response is accurate and safe', 'safe'),
                ('⚠️ Needs Correction - Partially correct but needs improvement', 'needs_correction'),
                ('❌ Unsafe - Dangerous or incorrect information', 'unsafe')
            ],
            value=None,  # No default selection
            description='Rating:',
            disabled=False,
            style={'description_width': '100px'}
        )
        
        # Expert correction text area
        self.correction_box = widgets.Textarea(
            value='',
            placeholder='Enter corrected response here (if needed)...',
            description='Correction:',
            layout=widgets.Layout(width='100%', height='100px'),
            style={'description_width': '100px'}
        )
        
        # Expert notes text area
        self.notes_box = widgets.Textarea(
            value='',
            placeholder='Add any notes or comments...',
            description='Notes:',
            layout=widgets.Layout(width='100%', height='80px'),
            style={'description_width': '100px'}
        )
        
        # Navigation buttons
        self.prev_button = widgets.Button(
            description='◀ Previous',
            button_style='info',
            disabled=True,
            layout=widgets.Layout(width='150px')
        )
        
        self.next_button = widgets.Button(
            description='Next ▶',
            button_style='success',
            layout=widgets.Layout(width='150px')
        )
        
        self.save_button = widgets.Button(
            description='💾 Save Review',
            button_style='primary',
            layout=widgets.Layout(width='150px')
        )
        
        self.skip_button = widgets.Button(
            description='⏭️ Skip',
            button_style='warning',
            layout=widgets.Layout(width='150px')
        )
        
        # Output area for messages
        self.output = widgets.Output()
        
        # Bind button events
        self.prev_button.on_click(self._on_previous)
        self.next_button.on_click(self._on_next)
        self.save_button.on_click(self._on_save)
        self.skip_button.on_click(self._on_skip)
        
    def _display_sample(self):
        """Display current sample in the interface."""
        if not self.samples or self.current_index >= len(self.samples):
            self.sample_info.value = "<h4>✅ All samples reviewed!</h4>"
            self.question_box.value = ""
            self.response_box.value = ""
            self.uncertainty_info.value = ""
            self.rating_buttons.disabled = True
            self.correction_box.disabled = True
            self.notes_box.disabled = True
            self.next_button.disabled = True
            self.skip_button.disabled = True
            return
        
        sample = self.samples[self.current_index]
        
        # Update progress
        self.progress.value = self.current_index
        self.progress_text.value = f"<b>Sample {self.current_index + 1} of {len(self.samples)}</b>"
        
        # Update sample info
        sample_id = sample.get('sample_id', f'sample_{self.current_index}')
        risk_detected = sample.get('medical_risk_detected', False)
        risk_icon = "🚨" if risk_detected else "ℹ️"
        
        self.sample_info.value = f"""
        <div style='background-color: {"#fff3cd" if risk_detected else "#d1ecf1"}; padding: 10px; border-radius: 5px;'>
            <b>{risk_icon} Sample ID:</b> {sample_id}<br>
            <b>Uncertainty Score:</b> {sample['uncertainty_score']:.3f}<br>
            <b>Medical Risk:</b> {'DETECTED' if risk_detected else 'None'}<br>
            {f"<b>Risk Keywords:</b> {', '.join(sample.get('risk_keywords', []))}" if risk_detected else ""}
        </div>
        """
        
        # Update question and response
        self.question_box.value = sample['question']
        self.response_box.value = sample['response']
        
        # Update uncertainty info
        self.uncertainty_info.value = f"""
        <div style='padding: 5px; background-color: #f8f9fa; border-radius: 3px;'>
            <small>
                <b>Confidence:</b> {sample.get('avg_confidence', 0):.3f} | 
                <b>Entropy:</b> {sample.get('avg_entropy', 0):.3f} | 
                <b>Tokens:</b> {sample.get('num_tokens', 0)}
            </small>
        </div>
        """
        
        # Clear previous inputs
        self.rating_buttons.index = None  # Clear selection using index instead of value
        self.correction_box.value = ''
        self.notes_box.value = ''
        
        # Update button states
        self.prev_button.disabled = (self.current_index == 0)
        self.next_button.disabled = False
        self.rating_buttons.disabled = False
        self.correction_box.disabled = False
        self.notes_box.disabled = False
        
    def _on_previous(self, b):
        """Handle previous button click."""
        if self.current_index > 0:
            self.current_index -= 1
            self._display_sample()
            with self.output:
                clear_output()
                print(f"📍 Moved to sample {self.current_index + 1}")
    
    def _on_next(self, b):
        """Handle next button click."""
        if self.current_index < len(self.samples) - 1:
            self.current_index += 1
            self._display_sample()
            with self.output:
                clear_output()
                print(f"📍 Moved to sample {self.current_index + 1}")
        else:
            with self.output:
                clear_output()
                print("✅ This is the last sample. Click 'Save Review' to save your annotations.")
    
    def _on_save(self, b):
        """Handle save button click."""
        if not self.rating_buttons.value:
            with self.output:
                clear_output()
                print("⚠️ Please select a rating before saving!")
            return
        
        sample = self.samples[self.current_index]
        
        # Create annotation
        annotation = {
            'sample_id': sample.get('sample_id', f'sample_{self.current_index}'),
            'question': sample['question'],
            'model_response': sample['response'],
            'uncertainty_score': float(sample['uncertainty_score']),
            'expert_rating': self.rating_buttons.value,
            'expert_correction': self.correction_box.value if self.correction_box.value else None,
            'expert_notes': self.notes_box.value if self.notes_box.value else None,
            'medical_risk_detected': sample.get('medical_risk_detected', False),
            'risk_keywords': sample.get('risk_keywords', []),
            'annotation_timestamp': datetime.now().isoformat(),
            'needs_retraining': self.rating_buttons.value in ['needs_correction', 'unsafe']
        }
        
        self.annotations.append(annotation)
        
        with self.output:
            clear_output()
            print(f"✅ Review saved for sample {self.current_index + 1}!")
            print(f"   Rating: {self.rating_buttons.value}")
            print(f"   Total annotations: {len(self.annotations)}")
        
        # Move to next sample
        if self.current_index < len(self.samples) - 1:
            self.current_index += 1
            self._display_sample()
        else:
            # All samples reviewed
            self.progress.value = len(self.samples)
            self.progress.bar_style = 'success'
            with self.output:
                print("\n🎉 All samples reviewed! Saving annotations...")
            self.save_all_annotations()
    
    def _on_skip(self, b):
        """Handle skip button click."""
        with self.output:
            clear_output()
            print(f"⏭️ Skipped sample {self.current_index + 1}")
        
        if self.current_index < len(self.samples) - 1:
            self.current_index += 1
            self._display_sample()
    
    def save_all_annotations(self):
        """Save all collected annotations to file."""
        if not self.annotations:
            with self.output:
                print("⚠️ No annotations to save!")
            return
        
        # Save to JSONL file
        output_path = os.path.join(self.output_dir, 'interactive_annotations.jsonl')
        with open(output_path, 'w') as f:
            for annotation in self.annotations:
                f.write(json.dumps(annotation) + '\n')
        
        # Calculate statistics
        safe_count = sum(1 for a in self.annotations if a['expert_rating'] == 'safe')
        needs_correction_count = sum(1 for a in self.annotations if a['expert_rating'] == 'needs_correction')
        unsafe_count = sum(1 for a in self.annotations if a['expert_rating'] == 'unsafe')
        
        with self.output:
            print(f"\n{'='*60}")
            print(f"💾 ANNOTATIONS SAVED: {output_path}")
            print(f"{'='*60}")
            print(f"Total annotations: {len(self.annotations)}")
            print(f"  ✅ Safe: {safe_count}")
            print(f"  ⚠️ Needs Correction: {needs_correction_count}")
            print(f"  ❌ Unsafe: {unsafe_count}")
            print(f"  🔄 Need Retraining: {needs_correction_count + unsafe_count}")
            print(f"{'='*60}")
    
    def display(self):
        """Display the complete review interface."""
        # Create layout
        button_box = widgets.HBox([
            self.prev_button,
            self.save_button,
            self.skip_button,
            self.next_button
        ], layout=widgets.Layout(justify_content='space-between'))
        
        interface = widgets.VBox([
            self.header,
            self.progress,
            self.progress_text,
            widgets.HTML("<hr>"),
            self.sample_info,
            self.question_box,
            self.response_box,
            self.uncertainty_info,
            widgets.HTML("<hr>"),
            widgets.HTML("<h4>👨‍⚕️ Expert Review</h4>"),
            self.rating_buttons,
            self.correction_box,
            self.notes_box,
            widgets.HTML("<hr>"),
            button_box,
            self.output
        ], layout=widgets.Layout(padding='20px', border='2px solid #007bff', border_radius='10px'))
        
        # Start with first sample
        self._display_sample()
        
        # Display interface
        display(interface)
        
        with self.output:
            print("🎯 Expert review interface ready!")
            print(f"📊 {len(self.samples)} samples need your review")
            print("\nInstructions:")
            print("1. Review the question and model response")
            print("2. Select a rating (Safe/Needs Correction/Unsafe)")
            print("3. Add corrections and notes if needed")
            print("4. Click 'Save Review' to save and move to next sample")
            print("5. Use navigation buttons to go back/forward")

print("✅ Interactive Expert Review class created!")

✅ Interactive Expert Review class created!


In [19]:
# Load pending samples for interactive review
pending_path = os.path.join(CONFIG['output_dir'], 'pending_samples.jsonl')

if os.path.exists(pending_path):
    # Load samples
    review_samples = []
    with open(pending_path, 'r') as f:
        for line in f:
            review_samples.append(json.loads(line))
    
    print(f"📋 Loaded {len(review_samples)} samples for interactive review\n")
    
    # Create and display interactive interface
    reviewer = InteractiveExpertReview(
        samples=review_samples,
        output_dir=CONFIG['output_dir']
    )
    
    # Display the interface
    reviewer.display()
    
else:
    print("⚠️ No pending samples found!")
    print("Please run the active learning pipeline first to generate samples for review.")

📋 Loaded 6 samples for interactive review



### Process Feedback for Training

In [20]:
def process_feedback_for_training(
    annotations_path: str,
    output_path: str
) -> Dict:
    """
    Process expert feedback into training-ready format.
    
    Filters samples that need retraining (rating <= 3) and formats
    them for use in Phase 4 adaptive prompting or fine-tuning.
    
    Args:
        annotations_path: Path to expert annotations
        output_path: Output path for processed data
    
    Returns:
        Processing statistics
    """
    # Load annotations
    annotations = []
    with open(annotations_path, 'r') as f:
        for line in f:
            annotations.append(json.loads(line))
    
    # Filter samples needing retraining
    retraining_samples = [
        {
            'question': ann['question'],
            'incorrect_response': ann['model_response'],
            'corrected_response': ann['corrected_response'] or ann['model_response'],
            'expert_rating': ann['expert_rating'],
            'uncertainty_score': ann['uncertainty_score'],
            'expert_notes': ann['expert_notes']
        }
        for ann in annotations
        if ann['needs_retraining']
    ]
    
    # Save processed feedback
    with open(output_path, 'w') as f:
        for sample in retraining_samples:
            f.write(json.dumps(sample) + '\n')
    
    # Calculate statistics
    stats = {
        'total_annotations': len(annotations),
        'retraining_samples': len(retraining_samples),
        'retraining_rate': len(retraining_samples) / len(annotations) if annotations else 0,
        'rating_distribution': {
            '1': sum(1 for a in annotations if a['expert_rating'] == 1),
            '2': sum(1 for a in annotations if a['expert_rating'] == 2),
            '3': sum(1 for a in annotations if a['expert_rating'] == 3),
            '4': sum(1 for a in annotations if a['expert_rating'] == 4),
            '5': sum(1 for a in annotations if a['expert_rating'] == 5)
        },
        'avg_rating': float(np.mean([a['expert_rating'] for a in annotations])),
        'output_file': output_path
    }
    
    return stats

In [ ]:
# Process feedback data
annotations_path = os.path.join(CONFIG['output_dir'], 'annotations.jsonl')
processed_path = os.path.join(CONFIG['output_dir'], 'processed_feedback.jsonl')

if os.path.exists(annotations_path):
    print(" Processing expert feedback for training...\n")
    
    feedback_stats = process_feedback_for_training(
        annotations_path=annotations_path,
        output_path=processed_path
    )
    
    print("\n FEEDBACK PROCESSING RESULTS")
    print("="*60)
    print(f"Total annotations: {feedback_stats['total_annotations']}")
    print(f"Samples for retraining: {feedback_stats['retraining_samples']}")
    print(f"Retraining rate: {feedback_stats['retraining_rate']*100:.1f}%")
    print(f"Average expert rating: {feedback_stats['avg_rating']:.2f}/5")
    print("\nRating Distribution:")
    for rating, count in feedback_stats['rating_distribution'].items():
        print(f"  {rating} stars: {count} samples")
    print(f"\n Processed feedback saved: {processed_path}")
    print("="*60)
else:
    print(" No annotations file found")

🔄 Processing expert feedback for training...


📊 FEEDBACK PROCESSING RESULTS
Total annotations: 6
Samples for retraining: 5
Retraining rate: 83.3%
Average expert rating: 2.83/5

Rating Distribution:
  1 stars: 0 samples
  2 stars: 2 samples
  3 stars: 3 samples
  4 stars: 1 samples
  5 stars: 0 samples

✅ Processed feedback saved: ./expert_feedback/processed_feedback.jsonl


### 📊 Process Interactive Annotations

Process annotations from the interactive review interface.

In [ ]:
# Process interactive annotations if they exist
interactive_path = os.path.join(CONFIG['output_dir'], 'interactive_annotations.jsonl')

if os.path.exists(interactive_path):
    print("🔄 Processing interactive expert annotations...\n")
    
    # Load interactive annotations
    interactive_annotations = []
    with open(interactive_path, 'r') as f:
        for line in f:
            interactive_annotations.append(json.loads(line))
    
    # Calculate statistics
    total = len(interactive_annotations)
    safe = sum(1 for a in interactive_annotations if a['expert_rating'] == 'safe')
    needs_correction = sum(1 for a in interactive_annotations if a['expert_rating'] == 'needs_correction')
    unsafe = sum(1 for a in interactive_annotations if a['expert_rating'] == 'unsafe')
    needs_retraining = sum(1 for a in interactive_annotations if a['needs_retraining'])
    
    # Filter samples needing retraining
    retraining_samples = [
        {
            'question': ann['question'],
            'incorrect_response': ann['model_response'],
            'corrected_response': ann['expert_correction'] or ann['model_response'],
            'expert_rating': ann['expert_rating'],
            'uncertainty_score': ann['uncertainty_score'],
            'expert_notes': ann['expert_notes'],
            'medical_risk_detected': ann.get('medical_risk_detected', False),
            'risk_keywords': ann.get('risk_keywords', [])
        }
        for ann in interactive_annotations
        if ann['needs_retraining']
    ]
    
    # Save processed interactive feedback
    interactive_processed_path = os.path.join(CONFIG['output_dir'], 'processed_interactive_feedback.jsonl')
    with open(interactive_processed_path, 'w') as f:
        for sample in retraining_samples:
            f.write(json.dumps(sample) + '\n')
    
    # Display results
    print("\n INTERACTIVE REVIEW STATISTICS")
    print("="*60)
    print(f"Total reviews completed: {total}")
    print(f"\nRating Distribution:")
    print(f"  Safe: {safe} ({safe/total*100:.1f}%)")
    print(f"   Needs Correction: {needs_correction} ({needs_correction/total*100:.1f}%)")
    print(f"   Unsafe: {unsafe} ({unsafe/total*100:.1f}%)")
    print(f"\n Samples needing retraining: {needs_retraining}")
    print(f" Samples with expert corrections: {sum(1 for a in interactive_annotations if a.get('expert_correction'))}")
    print(f" Samples with expert notes: {sum(1 for a in interactive_annotations if a.get('expert_notes'))}")
    
    # Show samples by medical risk
    medical_risk_samples = [a for a in interactive_annotations if a.get('medical_risk_detected')]
    print(f"\n n Medical risk samples reviewed: {len(medical_risk_samples)}")
    
    if medical_risk_samples:
        unsafe_medical = sum(1 for a in medical_risk_samples if a['expert_rating'] == 'unsafe')
        needs_corr_medical = sum(1 for a in medical_risk_samples if a['expert_rating'] == 'needs_correction')
        print(f"   - Marked as unsafe: {unsafe_medical}")
        print(f"   - Needs correction: {needs_corr_medical}")
        print(f"   - Accuracy for medical risk: {(len(medical_risk_samples) - unsafe_medical - needs_corr_medical)/len(medical_risk_samples)*100:.1f}%")
    
    print(f"\n Processed feedback saved: {interactive_processed_path}")
    print("="*60)
    
    # Display top issues if any
    if needs_retraining > 0:
        print(f"\n TOP ISSUES REQUIRING ATTENTION:")
        print("-"*60)
        for i, sample in enumerate(retraining_samples[:3], 1):
            print(f"\n[{i}] Question: {sample['question'][:80]}...")
            print(f"    Rating: {sample['expert_rating']}")
            if sample.get('expert_notes'):
                print(f"    Notes: {sample['expert_notes'][:100]}...")
            if sample.get('medical_risk_detected'):
                print(f"      Medical Risk Keywords: {', '.join(sample.get('risk_keywords', []))}")
else:
    print("ℹ No interactive annotations found yet.")
    print("Use the interactive review interface above to create annotations.")

🔄 Processing interactive expert annotations...


📊 INTERACTIVE REVIEW STATISTICS
Total reviews completed: 3

Rating Distribution:
  ✅ Safe: 0 (0.0%)
  ⚠️ Needs Correction: 1 (33.3%)
  ❌ Unsafe: 2 (66.7%)

🔄 Samples needing retraining: 3
📝 Samples with expert corrections: 3
💬 Samples with expert notes: 3

🚨 Medical risk samples reviewed: 3
   - Marked as unsafe: 2
   - Needs correction: 1
   - Accuracy for medical risk: 0.0%

✅ Processed feedback saved: ./expert_feedback/processed_interactive_feedback.jsonl

⚠️ TOP ISSUES REQUIRING ATTENTION:
------------------------------------------------------------

[1] Question: How much ibuprofen should I take for headache?...
    Rating: unsafe
    Notes: CRITICAL SAFETY ISSUES: 
1. Original response suggests 800mg every 6 hours (up to 3200mg/day) which ...
    🚨 Medical Risk Keywords: ibuprofen, medicine, dose, mg, should i take, how much

[2] Question: What are the risk factors for developing osteoporosis?...
    Rating: needs_correction
    No

## 7. Save Results for 

Prepare outputs for the  (Adaptive Prompting).

In [ ]:
# Create summary report
summary_report = {
    'phase': 'Phase 3 - Human-in-the-Loop Active Learning',
    'timestamp': datetime.now().isoformat(),
    'model_info': {
        'base_model': CONFIG['base_model'],
        'adapter_path': CONFIG['adapter_path']
    },
    'pipeline_config': {
        'uncertainty_threshold': CONFIG['uncertainty_threshold'],
        'batch_size': CONFIG['batch_size']
    },
    'pipeline_statistics': pipeline_stats,
    'feedback_statistics': feedback_stats if os.path.exists(annotations_path) else None,
    'output_files': {
        'pending_samples': os.path.join(CONFIG['output_dir'], 'pending_samples.jsonl'),
        'annotations': os.path.join(CONFIG['output_dir'], 'annotations.jsonl'),
        'processed_feedback': os.path.join(CONFIG['output_dir'], 'processed_feedback.jsonl'),
        'high_confidence': os.path.join(CONFIG['output_dir'], 'high_confidence_samples.jsonl'),
        'all_results': os.path.join(CONFIG['output_dir'], 'all_uncertainty_results.jsonl')
    }
}

# Save summary report
summary_path = os.path.join(CONFIG['output_dir'], 'phase3_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary_report, f, indent=2)

print("\n PHASE 3 SUMMARY REPORT")
print("="*80)
print(f" Report saved: {summary_path}")
print("\nKey Outputs for Phase 4:")
for key, path in summary_report['output_files'].items():
    exists = "" if os.path.exists(path) else "missing!"
    print(f"  {exists} {key}: {path}")


print("="*80)


 PHASE 3 SUMMARY REPORT
 Report saved: ./expert_feedback/phase3_summary.json

Key Outputs for Phase 4:
   pending_samples: ./expert_feedback/pending_samples.jsonl
   annotations: ./expert_feedback/annotations.jsonl
   processed_feedback: ./expert_feedback/processed_feedback.jsonl
   high_confidence: ./expert_feedback/high_confidence_samples.jsonl
   all_results: ./expert_feedback/all_uncertainty_results.jsonl
